# 02 — Positive Recording and Augmentation Pilot

A local, single-speaker experiment for **Hey GPT**. This is a pilot, not a finalized dataset pipeline.

1. Run the setup cells and the control-panel cell below using the project `.venv` kernel.
2. Click **Record positive**. Wait for the countdown, then say “Hey GPT” once, leaving a short pause before and after it. Repeat about ten times.
3. Optionally click **Record room noise** and remain silent for eight seconds.
4. Click **Generate 6 variants** for the latest recording, or **Augment session** to process all positives collected in this session. Files are named and saved automatically.

Start with a **2.0-second window**. The original recording is four seconds long and is always preserved. Automatic trimming only suggests a phrase boundary; listen before accepting it. If necessary, adjust the selection slider and preview it. A phrase that does not fit is rejected rather than truncated.

Audio stays local under `data/` and is ignored by Git. Microphone capture starts only when a recording button is clicked. **Run All does not record audio.**

Playback uses `normalize=False` so that gain differences remain audible. Keep the same speaker/headphone volume while comparing variants. Nothing is uploaded and no API is used.


In [ ]:
import json
import time
from datetime import datetime, timezone
from math import gcd
from pathlib import Path
from uuid import uuid4

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from audiomentations import AddBackgroundNoise, AddGaussianSNR, Gain, TimeStretch
from IPython.display import SVG, Audio, display
from scipy.signal import resample_poly

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the GPTWake project.")

SAMPLE_RATE = 16000
RECORD_SECONDS = 4
NOISE_SECONDS = 8
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:6]
RAW_DIR = ROOT / "data" / "raw" / "positive_pilot" / SESSION_ID
NOISE_DIR = ROOT / "data" / "negatives" / "room_noise_pilot" / SESSION_ID
AUG_DIR = ROOT / "data" / "processed" / "positive_pilot" / SESSION_ID
for directory in [RAW_DIR, NOISE_DIR, AUG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

state = {"audio": None, "source": None}
print("Session:", SESSION_ID)
print("Original WAV files:", RAW_DIR)
print("Augmented WAV files:", AUG_DIR)

In [ ]:
def write_wav(path, audio):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("Expected nonempty, finite mono audio.")
    peak = float(np.max(np.abs(audio)))
    if peak > 1.0:
        print(f"Warning: limiting peak {peak:.3f} to avoid PCM clipping: {path.name}")
        audio = audio * (0.999 / peak)
    sf.write(path, audio, SAMPLE_RATE, subtype="PCM_16")
    return audio


def capture(seconds, device):
    info = sd.query_devices(device, "input")
    rate = SAMPLE_RATE
    try:
        sd.check_input_settings(device=device, channels=1, samplerate=rate, dtype="float32")
    except (ValueError, sd.PortAudioError):
        rate = int(info["default_samplerate"])
        sd.check_input_settings(device=device, channels=1, samplerate=rate, dtype="float32")
    for remaining in [3, 2, 1]:
        print(f"Ready in {remaining}...", flush=True)
        time.sleep(1)
    print(f"RECORDING for {seconds} seconds...", flush=True)
    audio = sd.rec(
        round(seconds * rate), samplerate=rate, channels=1, dtype="float32", device=device
    )
    try:
        sd.wait()
    except BaseException:
        sd.stop()
        raise
    audio = audio[:, 0]
    input_peak = float(np.max(np.abs(audio)))
    if input_peak >= 0.99:
        print("Warning: possible input clipping. Speak more softly or lower microphone gain.")
    if rate != SAMPLE_RATE:
        factor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // factor, rate // factor)
    print("Recording complete.")
    return np.asarray(audio, dtype=np.float32), {
        "device": str(info["name"]),
        "capture_rate": rate,
        "input_peak": input_peak,
    }


def suggest_bounds(audio):
    frame = round(0.02 * SAMPLE_RATE)
    hop = round(0.01 * SAMPLE_RATE)
    starts = np.arange(0, max(1, len(audio) - frame + 1), hop)
    rms = np.array([np.sqrt(np.mean(audio[s : s + frame] ** 2)) for s in starts])
    threshold = max(0.001, float(np.percentile(rms, 20)) * 3, float(rms.max()) * 0.12)
    active = np.flatnonzero(rms > threshold)
    if not len(active):
        print("No clear phrase found. Check the microphone and adjust the selection manually.")
        return (0.0, len(audio) / SAMPLE_RATE)
    start = max(0, int(starts[active[0]]) - round(0.12 * SAMPLE_RATE))
    end = min(len(audio), int(starts[active[-1]]) + frame + round(0.15 * SAMPLE_RATE))
    return (start / SAMPLE_RATE, end / SAMPLE_RATE)


def show_audio(audio, title):
    peak = float(np.max(np.abs(audio)))
    print(f"{title}: {len(audio) / SAMPLE_RATE:.2f}s, peak={peak:.3f}")
    display(Audio(audio, rate=SAMPLE_RATE, normalize=False))
    # Display every bin's peak so short transients remain visible.
    bins = np.array_split(np.abs(audio), min(500, len(audio)))
    peaks = [float(b.max()) for b in bins]
    lines = "".join(
        f'<line x1="{i * 1.2:.1f}" x2="{i * 1.2:.1f}" '
        f'y1="{45 - v * 42:.1f}" y2="{45 + v * 42:.1f}" '
        'stroke="#2475c0"/>'
        for i, v in enumerate(peaks)
    )
    display(
        SVG(
            f'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="90" '
            f'viewBox="0 0 600 90"><rect width="600" height="90" fill="#f3f6fa"/>{lines}</svg>'
        )
    )


def random_place(phrase, window_seconds, rng):
    total = round(window_seconds * SAMPLE_RATE)
    if not 0 < len(phrase) <= total:
        raise ValueError(
            "The selected phrase does not fit. Adjust its bounds or increase the window."
        )
    offset = int(rng.integers(0, total - len(phrase) + 1))
    result = np.zeros(total, dtype=np.float32)
    result[offset : offset + len(phrase)] = phrase
    return result, offset / SAMPLE_RATE


def make_variants(phrase, window_seconds, noise_dir, seed):
    rng = np.random.default_rng(seed)
    if len(phrase) > round(window_seconds * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window; no positive was truncated. Adjust selection or window."
        )
    variants = []
    for i in range(3):
        audio, offset = random_place(phrase, window_seconds, rng)
        variants.append((f"placement_{i + 1}", audio, {"offset_seconds": offset}))
    quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
    quieter = quiet(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(quieter, window_seconds, rng)
    variants.append(("quieter", audio, {"offset_seconds": offset, **quiet.parameters}))
    audio, offset = random_place(phrase, window_seconds, rng)
    if list(noise_dir.glob("*.wav")):
        noise = AddBackgroundNoise(sounds_path=noise_dir, min_snr_db=10, max_snr_db=25, p=1)
        label = "room_noise"
    else:
        noise = AddGaussianSNR(min_snr_db=10, max_snr_db=25, p=1)
        label = "gaussian_noise_demo"
    noisy = noise(samples=audio.copy(), sample_rate=SAMPLE_RATE)
    variants.append((label, noisy, {"offset_seconds": offset, **noise.parameters}))
    # Faster speech fits whenever the original phrase fits. Preserve pitch and do not pad/crop it here.
    stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=False, p=1)
    faster = stretch(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(faster, window_seconds, rng)
    variants.append(("faster", audio, {"offset_seconds": offset, **stretch.parameters}))
    return variants

In [ ]:
input_options = [("System default microphone", None)] + [
    (f"{i}: {d['name']}", i)
    for i, d in enumerate(sd.query_devices())
    if d["max_input_channels"] > 0
]
device = widgets.Dropdown(
    options=input_options, description="Microphone:", layout={"width": "650px"}
)
window = widgets.Dropdown(options=[1.5, 2.0, 2.5, 3.0], value=2.0, description="Window (s):")
selection = widgets.FloatRangeSlider(
    value=(0, RECORD_SECONDS),
    min=0,
    max=RECORD_SECONDS,
    step=0.01,
    description="Phrase (s):",
    readout_format=".2f",
    continuous_update=False,
    layout={"width": "650px"},
)
record_button = widgets.Button(description="Record positive", button_style="success")
noise_button = widgets.Button(description="Record room noise")
preview_button = widgets.Button(description="Preview selection")
augment_button = widgets.Button(description="Generate 6 variants", button_style="info")
batch_button = widgets.Button(description="Augment session")
output = widgets.Output()
buttons = [record_button, noise_button, preview_button, augment_button, batch_button]


def run_action(action):
    for button in buttons:
        button.disabled = True
    try:
        with output:
            output.clear_output(wait=True)
            action()
    except Exception as error:
        with output:
            print(f"{type(error).__name__}: {error}")
    finally:
        for button in buttons:
            button.disabled = False


def record_positive():
    audio, metadata = capture(RECORD_SECONDS, device.value)
    path = RAW_DIR / f"positive_{uuid4().hex[:10]}.wav"
    audio = write_wav(path, audio)
    path.with_suffix(".json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    state.update(audio=audio, source=path)
    selection.value = suggest_bounds(audio)
    print("Saved original:", path)
    print("Suggested phrase bounds:", selection.value, "— check them by listening.")
    show_audio(audio, "Original (untouched timing)")


def record_noise():
    print("Remain silent during capture; keep normal room sounds.")
    audio, metadata = capture(NOISE_SECONDS, device.value)
    path = NOISE_DIR / f"room_{uuid4().hex[:10]}.wav"
    write_wav(path, audio)
    path.with_suffix(".json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("Saved room noise:", path)
    show_audio(audio, "Room noise")


def selected_phrase():
    if state["audio"] is None:
        raise ValueError("Record a positive sample first.")
    start, end = selection.value
    phrase = state["audio"][round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)]
    if not len(phrase):
        raise ValueError("Select a nonempty phrase interval.")
    return phrase


def preview_selection():
    show_audio(selected_phrase(), "Selected phrase — ensure the entire Hey GPT is audible")


def generate_variants(show_players=True):
    phrase = selected_phrase()
    seed = uuid4().int % (2**32)
    variants = make_variants(phrase, window.value, NOISE_DIR, seed)
    folder = AUG_DIR / (state["source"].stem + "_" + uuid4().hex[:6])
    folder.mkdir(parents=True, exist_ok=False)
    write_wav(folder / "selected_phrase.wav", phrase)
    metadata = {
        "source": str(state["source"].relative_to(ROOT)),
        "selection_seconds": selection.value,
        "window_seconds": window.value,
        "placement_seed": seed,
        "sample_rate": SAMPLE_RATE,
        "variants": [],
    }
    print("Saved variants:", folder)
    print(
        "Placement uses a recorded seed; other transform draws are documented by their parameters."
    )
    if not list(NOISE_DIR.glob("*.wav")):
        print("No room noise recorded: using Gaussian noise for demonstration only.")
    if show_players:
        show_audio(phrase, "Selected original")
    for name, audio, params in variants:
        audio = write_wav(folder / f"{name}.wav", audio)
        metadata["variants"].append({"file": f"{name}.wav", "parameters": params})
        if show_players:
            show_audio(audio, name)
    (folder / "metadata.json").write_text(
        json.dumps(
            metadata, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)
        ),
        encoding="utf-8",
    )


def augment_session():
    paths = sorted(RAW_DIR.glob("positive_*.wav"))
    if not paths:
        raise ValueError("Record a positive sample first.")
    saved_state = state.copy()
    saved_selection = selection.value
    processed = 0
    try:
        for path in paths:
            audio, rate = sf.read(path, dtype="float32")
            if rate != SAMPLE_RATE:
                raise ValueError(f"Unexpected sample rate in {path.name}")
            state.update(audio=audio, source=path)
            selection.value = suggest_bounds(audio)
            if len(selected_phrase()) > round(window.value * SAMPLE_RATE):
                print(f"Skipped {path.name}: suggested phrase exceeds the window; review manually.")
                continue
            generate_variants(show_players=False)
            processed += 1
    finally:
        state.update(saved_state)
        selection.value = saved_selection
    print(
        f"Processed {processed}/{len(paths)} originals. Batch trimming is provisional; review the saved audio."
    )


record_button.on_click(lambda _: run_action(record_positive))
noise_button.on_click(lambda _: run_action(record_noise))
preview_button.on_click(lambda _: run_action(preview_selection))
augment_button.on_click(lambda _: run_action(generate_variants))
batch_button.on_click(lambda _: run_action(augment_session))
display(widgets.VBox([device, window, selection, widgets.HBox(buttons), output]))
print("Ready. Click Record positive when you want to capture audio.")

## What to check

- Listen to the original and the selected phrase: automatic boundaries must not cut off “Hey” or the final “T”. The energy-based suggestion is not a speech recognizer and may include keyboard or room sounds.
- Every generated window is exactly the selected duration. Phrase placement changes without wraparound or truncation.
- The three placement examples may occasionally be close together; offsets are sampled randomly and recorded in metadata.
- The faster example uses pitch-preserving time stretching. Gain and noise variations remain audible because playback does not normalize each clip.
- If the microphone is wrong, select another input from the dropdown. Persistent silence or clipping should be corrected before collecting more recordings.
- This single-speaker pilot checks the workflow only. It does not measure generalization to other speakers. Split real data by speaker and source recording before generating training variants.

Files are automatically saved; closing the notebook does not delete them. Re-running setup creates a new session, leaving earlier recordings intact. Save the notebook with outputs cleared before committing it.
